# Analisis Exp2 Reestructurado

Carga de datos (episodios=1000), res?menes por agente/riesgo y visualizaciones b?sicas.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style='whitegrid')

candidatos = [Path('..')/'data'/'master_results_combined.csv', Path('..')/'data'/'master_results_clean.csv']
for c in candidatos:
    if c.exists():
        df = pd.read_csv(c)
        print(f'Archivo cargado: {c}')
        break
else:
    raise FileNotFoundError('No se encontro master_results_combined.csv ni master_results_clean.csv en ../data')

if 'episodes' in df.columns:
    df = df[df['episodes'] == 1000]

print('Columnas:', df.columns.tolist())
print('Primeras filas:')
display(df.head())

In [ ]:
# Conteos basicos
if {'agent','risk_scale'}.issubset(df.columns):
    print('\nConteo por agente y riesgo:')
    display(df.groupby(['agent','risk_scale']).size())
if 'seed' in df.columns:
    print('\nSeeds unicas:', sorted(df['seed'].dropna().unique()))

In [ ]:
# Resumen de recompensas y tripwires por agente/riesgo
if {'agent','risk_scale','reward','tripwires'}.issubset(df.columns):
    resumen = df.groupby(['agent','risk_scale'])[['reward','tripwires']].agg(['mean','std','count']).round(3)
    display(resumen.head(30))
    pivot_reward = resumen['reward']['mean'].unstack()
    print('\nReward medio por agente/riesgo:')
    display(pivot_reward)
else:
    print('Faltan columnas reward/tripwires para el resumen.')

In [ ]:
# Visualizaciones
if {'agent','reward'}.issubset(df.columns):
    plt.figure(figsize=(9,5))
    sns.boxplot(x='agent', y='reward', data=df, showfliers=False)
    plt.title('Distribucion de recompensas por agente (sin outliers)')
    plt.show()
else:
    print('No se puede graficar boxplot: faltan agent/reward')

if {'agent','risk_scale','reward'}.issubset(df.columns):
    plt.figure(figsize=(9,5))
    sns.lineplot(data=df, x='risk_scale', y='reward', hue='agent', estimator='mean', errorbar='sd', marker='o')
    plt.title('Reward medio vs riesgo por agente')
    plt.show()

if {'agent','risk_scale','tripwires'}.issubset(df.columns):
    plt.figure(figsize=(9,5))
    sns.barplot(data=df, x='risk_scale', y='tripwires', hue='agent', estimator='mean', errorbar='sd')
    plt.title('Tripwires medios vs riesgo por agente')
    plt.show()